In [ ]:
!pip install accelerate peft bitsandbytes transformers trl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 18.5 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset, Dataset
from peft import LoraConfig, AutoPeftModelForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer
import os

In [ ]:
dataset = load_dataset("json", data_files=r"D:\DSA project\formatted_tinyllama_dataset.json")

# Model and output paths
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
output_model = "tinyllama-improm2prompt-v1"

FileNotFoundError: Unable to find 'D:\DSA project\formatted_tinyllama_dataset.json'

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving formatted_tinyllama_dataset.json to formatted_tinyllama_dataset.json


In [ ]:
dataset = load_dataset("json", data_files="formatted_tinyllama_dataset.json")


Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
output_model = "tinyllama-improm2prompt-v1"


In [ ]:
def get_model_and_tokenizer(model_id):
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    # 4-bit quantization setup for QLoRA
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,  # better stability
        bnb_4bit_use_double_quant=True,
    )

    # Load model with quantization
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Configuration tweaks for fine-tuning
    model.config.use_cache = False
    model.config.pretraining_tp = 1

    # Enable gradient checkpointing to save memory
    model.gradient_checkpointing_enable()

    return model, tokenizer

In [ ]:
model, tokenizer = get_model_and_tokenizer(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
print("Tokenizer loaded ✅")
print("Pad token:", tokenizer.pad_token)
print("Vocabulary size:", tokenizer.vocab_size)


Tokenizer loaded ✅
Pad token: </s>
Vocabulary size: 32000


In [ ]:


peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],  # focus LoRA on attention
)


In [ ]:
from transformers import TrainingArguments

training_arguments = TrainingArguments(
    output_dir=output_model,
    per_device_train_batch_size=4,       # safer for Colab GPUs
    gradient_accumulation_steps=2,       # effective batch size = 8
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,                           # mixed precision
    report_to="none",
)


In [ ]:
import trl
print(trl.__version__)



0.24.0


In [ ]:
!pip install -U -q \
  transformers==4.44.2 \
  trl==0.9.6 \
  peft==0.10.0 \
  accelerate==0.33.0 \
  bitsandbytes==0.43.1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requir

In [ ]:
import torch, transformers, trl, peft, accelerate, bitsandbytes
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)


Torch: 2.8.0+cu126
Transformers: 4.57.1
TRL: 0.24.0
PEFT: 0.17.1
Accelerate: 1.11.0
BitsAndBytes: 0.48.2


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

ModuleNotFoundError: No module named 'triton.ops'

In [ ]:
!pip uninstall -y bitsandbytes triton
!pip install -U bitsandbytes==0.43.3 triton==2.3.0


Found existing installation: bitsandbytes 0.43.1
Uninstalling bitsandbytes-0.43.1:
  Successfully uninstalled bitsandbytes-0.43.1
Found existing installation: triton 3.4.0
Uninstalling triton-3.4.0:
  Successfully uninstalled triton-3.4.0
  Using cached bitsandbytes-0.43.3-py3-none-manylinux_2_24_x86_64.whl.metadata (3.5 kB)
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 73.1 M

In [ ]:
import bitsandbytes as bnb
print("BitsAndBytes GPU available:", bnb.cuda_available)


AttributeError: module 'bitsandbytes' has no attribute 'cuda_available'

In [ ]:
!pip uninstall -y bitsandbytes
!rm -rf /usr/local/lib/python3.*/dist-packages/bitsandbytes
!rm -rf /usr/lib/python3.*/dist-packages/bitsandbytes


Found existing installation: bitsandbytes 0.43.3
Uninstalling bitsandbytes-0.43.3:
  Successfully uninstalled bitsandbytes-0.43.3


In [ ]:
!pip install bitsandbytes-gpu

ERROR: Could not find a version that satisfies the requirement bitsandbytes-gpu (from versions: none)
ERROR: No matching distribution found for bitsandbytes-gpu


In [ ]:
!pip install -U bitsandbytes-cuda118==0.43.3


ERROR: Could not find a version that satisfies the requirement bitsandbytes-cuda118==0.43.3 (from versions: none)
ERROR: No matching distribution found for bitsandbytes-cuda118==0.43.3


In [ ]:
!pip uninstall -y bitsandbytes
!rm -rf /usr/local/lib/python3*/dist-packages/bitsandbytes
!rm -rf /usr/lib/python3*/dist-packages/bitsandbytes


In [ ]:
!pip uninstall -y bitsandbytes
!rm -rf /usr/local/lib/python*/dist-packages/bitsandbytes*
!rm -rf /usr/lib/python*/dist-packages/bitsandbytes*


In [ ]:
!pip install bitsandbytes==0.43.3


  Using cached bitsandbytes-0.43.3-py3-none-manylinux_2_24_x86_64.whl.metadata (3.5 kB)
Using cached bitsandbytes-0.43.3-py3-none-manylinux_2_24_x86_64.whl (137.5 MB)


In [ ]:
import torch, bitsandbytes as bnb
print("Torch CUDA available:", torch.cuda.is_available())

try:
    from bitsandbytes import cuda_setup
    print(cuda_setup.main_check())
except Exception as e:
    print("bitsandbytes check error:", e)


Torch CUDA available: False
bitsandbytes check error: cannot import name 'cuda_setup' from 'bitsandbytes' (/usr/local/lib/python3.12/dist-packages/bitsandbytes/__init__.py)


In [ ]:
!nvidia-smi


Tue Nov  4 21:13:38 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -U bitsandbytes==0.43.3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 7.7 MB/s eta 0:00:00


In [ ]:
import torch, bitsandbytes as bnb
print("Torch CUDA available:", torch.cuda.is_available())


ModuleNotFoundError: No module named 'triton.ops'

In [ ]:
!pip uninstall -y bitsandbytes triton
!rm -rf /usr/local/lib/python3*/dist-packages/bitsandbytes*
!rm -rf /usr/local/lib/python3*/dist-packages/triton*


Found existing installation: bitsandbytes 0.43.3
Uninstalling bitsandbytes-0.43.3:
  Successfully uninstalled bitsandbytes-0.43.3
Found existing installation: triton 3.4.0
Uninstalling triton-3.4.0:
  Successfully uninstalled triton-3.4.0


In [ ]:
!pip install bitsandbytes==0.43.3 triton==3.0.0


  Using cached bitsandbytes-0.43.3-py3-none-manylinux_2_24_x86_64.whl.metadata (3.5 kB)
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
Using cached bitsandbytes-0.43.3-py3-none-manylinux_2_24_x86_64.whl (137.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 114.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2

In [ ]:
import torch, bitsandbytes as bnb
print("Torch CUDA available:", torch.cuda.is_available())

try:
    import bitsandbytes.cuda_setup as cuda_setup
    print(cuda_setup.main_check())
except Exception as e:
    print("bitsandbytes check error:", e)


Torch CUDA available: True
bitsandbytes check error: No module named 'bitsandbytes.cuda_setup'


In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(load_in_4bit=True)
print("BitsAndBytes 4-bit enabled:", bnb_config.load_in_4bit)
print("GPU device:", torch.cuda.get_device_name(0))


BitsAndBytes 4-bit enabled: True
GPU device: Tesla T4


In [ ]:
# === Core imports for LoRA fine-tuning ===
from peft import LoraConfig
from trl import SFTTrainer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from datasets import load_dataset
import torch


ModuleNotFoundError: Could not import module 'PreTrainedModel'. Are this object's requirements defined correctly?

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1
!pip install -U transformers==4.44.2 trl==0.9.6 peft==0.10.0 accelerate==0.33.0 bitsandbytes==0.43.3


Found existing installation: torch 2.4.1
Uninstalling torch-2.4.1:
  Successfully uninstalled torch-2.4.1
Found existing installation: torchvision 0.23.0+cu126
Uninstalling torchvision-0.23.0+cu126:
  Successfully uninstalled torchvision-0.23.0+cu126
Found existing installation: torchaudio 2.8.0+cu126
Uninstalling torchaudio-2.8.0+cu126:
  Successfully uninstalled torchaudio-2.8.0+cu126
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cudnn-cu12
    Found existing installation: nvidia-cudnn-cu12 9.1.0.70
    Uninstalling nvidia-cudnn-cu12-9.1.0.70:
      Successfully uninstalled nvidia-cudnn-cu12-9.1.0.70


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 13.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Succ

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

print("Torch version:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("GPU available:", torch.cuda.is_available())


Torch version: 2.3.1+cu121
Transformers: 4.44.2
GPU available: True


In [ ]:
# === Dataset ===
from datasets import load_dataset

dataset = load_dataset("json", data_files="formatted_tinyllama_dataset.json")
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

# === Model + Tokenizer ===
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
output_model = "tinyllama-improm2prompt-v1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# === LoRA Configuration ===
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

# === Training Arguments ===
from transformers import TrainingArguments

training_arguments = TrainingArguments(
    output_dir=output_model,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    report_to="none",
)

# === SFT Trainer ===
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    args=training_arguments,
    packing=False,
    max_seq_length=1024,
)

# === Start Training ===
trainer.train()

# === Save the Fine-tuned Model ===
trainer.save_model(output_model)
print(f"✅ Training complete. Model saved to {output_model}")


FileNotFoundError: Unable to find '/content/formatted_tinyllama_dataset.json'

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving formatted_tinyllama_dataset.json to formatted_tinyllama_dataset.json


In [ ]:
# === Dataset ===
from datasets import load_dataset

dataset = load_dataset("json", data_files="formatted_tinyllama_dataset.json")
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

# === Model + Tokenizer ===
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
output_model = "tinyllama-improm2prompt-v1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# === LoRA Configuration ===
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

# === Training Arguments ===
from transformers import TrainingArguments

training_arguments = TrainingArguments(
    output_dir=output_model,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    report_to="none",
)

# === SFT Trainer ===
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    args=training_arguments,
    packing=False,
    max_seq_length=1024,
)

# === Start Training ===
trainer.train()

# === Save the Fine-tuned Model ===
trainer.save_model(output_model)
print(f"✅ Training complete. Model saved to {output_model}")


Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/432 [00:00<?, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Step,Training Loss
10,3.053500
20,2.067900
30,1.072900
40,0.734100
50,0.613500
60,0.459600
70,0.411300
80,0.351000
90,0.365700
100,0.327200


✅ Training complete. Model saved to tinyllama-improm2prompt-v1


In [ ]:
from transformers import pipeline

# Load the fine-tuned model and tokenizer
model_dir = "tinyllama-improm2prompt-v1"

pipe = pipeline(
    "text-generation",
    model=model_dir,
    tokenizer=model_dir,
    torch_dtype=torch.float16,
    device_map="auto",
)

# === Test Prompts ===
prompts = [
    "Explain the concept of a stack data structure.",
    "Refine the following query into a clear instruction: how to find the shortest path in a graph?",
    "Describe the difference between merge sort and quick sort.",
]

for i, prompt in enumerate(prompts, 1):
    print(f"\n🧠 Prompt {i}: {prompt}\n")
    output = pipe(prompt, max_new_tokens=150, do_sample=True, temperature=0.7, top_p=0.9)
    print("💬 Model Response:", output[0]["generated_text"])



🧠 Prompt 1: Explain the concept of a stack data structure.

💬 Model Response: Explain the concept of a stack data structure.

🧠 Prompt 2: Refine the following query into a clear instruction: how to find the shortest path in a graph?

💬 Model Response: Refine the following query into a clear instruction: how to find the shortest path in a graph?

🧠 Prompt 3: Describe the difference between merge sort and quick sort.

💬 Model Response: Describe the difference between merge sort and quick sort.


In [ ]:
from transformers import pipeline

model_dir = "tinyllama-improm2prompt-v1"

pipe = pipeline(
    "text-generation",
    model=model_dir,
    tokenizer=model_dir,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Helper: format prompt same way as training
def format_prompt(user_input):
    return f"<|user|>\n{user_input}</s>\n<|assistant|>\n"

# Test prompts
prompts = [
    "Explain the concept of a stack data structure.",
    "Refine the following query into a clear instruction: how to find the shortest path in a graph?",
    "Describe the difference between merge sort and quick sort.",
]

for i, prompt in enumerate(prompts, 1):
    print(f"\n🧠 Prompt {i}: {prompt}\n")
    formatted = format_prompt(prompt)
    output = pipe(formatted, max_new_tokens=150, do_sample=True, temperature=0.7, top_p=0.9)
    response = output[0]["generated_text"].split("<|assistant|>")[-1].strip()
    print("💬 Model Response:", response)



🧠 Prompt 1: Explain the concept of a stack data structure.

💬 Model Response: A stack data structure is a data structure that maintains a list of elements in reverse order. It is similar to a queue in that it allows elements to be added to the top of the stack and removed from the top. However, unlike a queue, which is always in a fixed position, a stack can be in any position in the list.

Here's how a stack works:

1. Create an empty stack.
2. Add an element to the top of the stack.
3. Repeat steps 2 and 3 until the stack is empty.
4. Remove the top element from the stack.

Here's an example of how a stack works:

```
stack = []

🧠 Prompt 2: Refine the following query into a clear instruction: how to find the shortest path in a graph?

💬 Model Response: To find the shortest path in a graph, you can use the Dijkstra's algorithm. Here's how to implement it in Python:

1. Define the graph data structure:

```python
class Graph:
    def __init__(self, adjacency_matrix):
        self.adj

In [ ]:
from transformers import pipeline

model_dir = "tinyllama-improm2prompt-v1"

pipe = pipeline(
    "text-generation",
    model=model_dir,
    tokenizer=model_dir,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Function to generate refined prompts
def refine_prompt(raw_prompt):
    # Match the same structure as in training
    formatted = (
        "### Instruction:\n"
        "Refine the following user prompt into a precise, structured, and detailed version.\n\n"
        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )

    output = pipe(
        formatted,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=pipe.tokenizer.eos_token_id,
    )[0]["generated_text"]

    # Extract only the refined part
    if "### Output:" in output:
        refined = output.split("### Output:")[-1].strip()
    else:
        refined = output.strip()

    return refined

# === Test it ===
test_prompts = [
    "rag system",
    "difference between ai and ml",
    "importance of tokenization",
    "generate creative writing ideas",
    "how to fine-tune a model",
]

for p in test_prompts:
    refined = refine_prompt(p)
    print(f"\n🧠 Original: {p}\n✨ Refined:  {refined}\n{'-'*80}")



🧠 Original: rag system
✨ Refined:  Enter the number of rows and columns for the grid:
Please enter the number of rows:
Please enter the number of columns:
--------------------------------------------------------------------------------

🧠 Original: difference between ai and ml
✨ Refined:  the percentage difference between ai and ml in terms of job opportunities in the united states in 2020
--------------------------------------------------------------------------------

🧠 Original: importance of tokenization
✨ Refined:  - Implement tokenization using a specified algorithm
- Output the processed text in a prettified format (e.g. Indented and formatted)
- Optionally, add a feature that converts all occurrences of the specified word to uppercase.
--------------------------------------------------------------------------------

🧠 Original: generate creative writing ideas
✨ Refined:  1. Start by brainstorming a list of creative writing prompts and exercises that can help you get started. 2

In [ ]:
from transformers import pipeline
import torch

model_dir = "tinyllama-improm2prompt-v1"
pipe = pipeline(
    "text-generation",
    model=model_dir,
    tokenizer=model_dir,
    torch_dtype=torch.float16,
    device_map="auto",
)

def refine_prompt(raw_prompt, max_new_tokens=120):
    template = (
        "### Instruction:\n"
        "Refine the following user prompt into a precise, structured, and detailed version.\n\n"
        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )
    # Deterministic / higher-quality generation: beam search, no sampling
    out = pipe(
        template,
        max_new_tokens=max_new_tokens,
        do_sample=False,        # disable sampling
        num_beams=4,           # beam search
        early_stopping=True,
        return_full_text=False
    )[0]["generated_text"]

    # robust extraction: after "### Output:" until next section or EOS
    if "### Output:" in out:
        refined = out.split("### Output:")[-1].strip()
    else:
        # fallback: remove the prompt prefix
        refined = out.replace(template, "").strip()
    # stop at next "### Instruction" or after newline markers possibly added
    refined = refined.split("### Instruction:")[0].strip()
    return refined

tests = [
    "rag system",
    "difference between ai and ml",
    "importance of tokenization"
]
for t in tests:
    print("RAW:", t)
    print("REFINED:", refine_prompt(t))
    print("-"*60)


RAW: rag system
REFINED: please enter the name of the rags to be cleaned
please enter the number of rags to be cleaned
please enter the number of rags to be washed
please enter the number of rags to be dried
please enter the number of rags to be folded
please enter the number of rags to be ironed
please enter the number of rags to be pressed
please enter the number of rags to be folded
please enter the number of rags to be ironed
please enter
------------------------------------------------------------
RAW: difference between ai and ml
REFINED: the difference between artificial intelligence (ai) and machine learning (ml)

### Explanation:
- ai: abbreviation for artificial intelligence
- ml: abbreviation for machine learning
- difference: the difference between ai and ml
------------------------------------------------------------
RAW: importance of tokenization
REFINED: Input: Importance of tokenization
------------------------------------------------------------


In [ ]:
import json, random, re
path = "formatted_tinyllama_dataset.json"
with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Quick checks:
print("Total entries:", len(data))

# 1) find exact duplicates of input==output (those teach identity/equality)
dupe_same = [i for i, e in enumerate(data) if "### Input:" in e["text"] and e["text"].split("### Input:")[-1].strip()[:100] == e["text"].split("### Output:")[-1].strip()[:100]]
print("examples where Input and Output start the same (count):", len(dupe_same))

# 2) find outputs that look like UI prompts or ask for number inputs
bad_patterns = []
for i,e in enumerate(data):
    out = e["text"].split("### Output:")[-1].strip().lower()
    if re.search(r"please enter|enter the number|prompt:\s*enter|please provide your|press enter", out):
        bad_patterns.append((i,out[:150]))
print("ui-like outputs found:", len(bad_patterns))
print(bad_patterns[:10])

# 3) sample some entries for manual inspection
print("\nRandom samples for manual check:")
for i in random.sample(range(len(data)), min(10, len(data))):
    print("----")
    print(data[i]["text"][:300])


Total entries: 480
examples where Input and Output start the same (count): 0
ui-like outputs found: 0
[]

Random samples for manual check:
----
### Instruction:
Refine the following user prompt into a precise, structured, and detailed version.

### Input:
compare writing tones

### Output:
Provide a detailed and technically accurate explanation of compare writing tones, including its principles, examples, and real-world relevance in creativ
----
### Instruction:
Refine the following user prompt into a precise, structured, and detailed version.

### Input:
Fenwick tree example

### Output:
Explain comprehensively of Fenwick tree, including its principles, examples, and real-world relevance in dsa applications.
----
### Instruction:
Refine the following user prompt into a precise, structured, and detailed version.

### Input:
importance of dataset curation

### Output:
Provide a detailed explanation of 'importance of dataset curation', including its principles, use cases, and relevance i

In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model="tinyllama-improm2prompt-v1",
    tokenizer="tinyllama-improm2prompt-v1",
    torch_dtype=torch.float16,
    device_map="auto",
)

def refine_prompt(raw_prompt):
    template = (
        "### Instruction:\n"
        "Refine the following user prompt into a precise, structured, and detailed version.\n\n"
        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )
    output = pipe(
        template,
        max_new_tokens=80,
        do_sample=False,       # turn off randomness
        num_beams=5,           # beam search
        early_stopping=True,
        eos_token_id=pipe.tokenizer.eos_token_id,
        return_full_text=False,
    )[0]["generated_text"]

    # Extract just the output portion
    refined = output.split("### Output:")[-1].strip()
    refined = refined.split("### Instruction:")[0].strip()
    return refined

# Test
tests = [
    "rag system",
    "difference between ai and ml",
    "importance of tokenization",
    "generate creative writing ideas",
    "how to fine-tune a model"
]

for t in tests:
    print(f"\n🧠 Original: {t}\n✨ Refined: {refine_prompt(t)}\n{'-'*80}")



🧠 Original: rag system
✨ Refined: please enter the name of the rags to be cleaned
please enter the number of rags to be cleaned
please enter the number of rags to be washed
please enter the number of rags to be dried
please enter the number of rags to be folded
please enter the number of rags to be ironed
ple
--------------------------------------------------------------------------------

🧠 Original: difference between ai and ml
✨ Refined: the difference between artificial intelligence (ai) and machine learning (ml)

### Explanation:
the difference between ai and ml is that ai is a branch of computer science that focuses on the development of intelligent machines, while ml is a branch of computer science that focuses on the development of machines that can learn and make decisions based on data.
--------------------------------------------------------------------------------

🧠 Original: importance of tokenization
✨ Refined: Input: Importance of tokenization
-------------------------

In [ ]:
training_arguments = TrainingArguments(
    output_dir=output_model,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,     # lower LR for better alignment
    num_train_epochs=5,     # 2 extra epochs
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
)


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    args=training_arguments,
    packing=False,
    max_seq_length=1024,
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instan

RuntimeError: WandbCallback requires wandb to be installed. Run `pip install wandb`.

In [ ]:
trainer.train()


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:


Abort: 

In [ ]:
!pip install -q wandb


In [ ]:
from trl import SFTTrainer

training_arguments = TrainingArguments(
    output_dir=output_model,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    num_train_epochs=5,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",   # ✅ disables wandb cleanly
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    args=training_arguments,
    packing=False,
    max_seq_length=1024,
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Step,Training Loss
10,2.961100
20,2.321900
30,1.638200
40,1.133700
50,0.839900
60,0.676200
70,0.632800
80,0.558100
90,0.553700
100,0.489900


TrainOutput(global_step=270, training_loss=0.6568045112821791, metrics={'train_runtime': 125.8018, 'train_samples_per_second': 17.17, 'train_steps_per_second': 2.146, 'total_flos': 1004650345660416.0, 'train_loss': 0.6568045112821791, 'epoch': 5.0})

In [ ]:
trainer.save_model(output_model)
print(f"✅ Model saved to {output_model}")


✅ Model saved to tinyllama-improm2prompt-v1


In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model="tinyllama-improm2prompt-v1",
    tokenizer="tinyllama-improm2prompt-v1",
    torch_dtype=torch.float16,
    device_map="auto",
)

def refine_prompt(raw_prompt):
    template = (
        "### Instruction:\n"
        "Refine the following user prompt into a precise, structured, and detailed version.\n\n"
        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )
    output = pipe(
        template,
        max_new_tokens=80,
        do_sample=False,   # deterministic
        num_beams=5,
        early_stopping=True,
        eos_token_id=pipe.tokenizer.eos_token_id,
        return_full_text=False,
    )[0]["generated_text"]
    refined = output.split("### Output:")[-1].strip().split("### Instruction:")[0].strip()
    return refined

# Try a few
prompts = [
    "rag system",
    "difference between ai and ml",
    "importance of tokenization",
    "generate creative writing ideas",
    "how to fine-tune a model",
]

for p in prompts:
    print(f"\n🧠 Original: {p}\n✨ Refined:  {refine_prompt(p)}\n{'-'*80}")



🧠 Original: rag system
✨ Refined:  please enter the name of the rags to be cleaned
please enter the number of rags to be cleaned
please enter the number of rags to be washed
please enter the number of rags to be dried
please enter the number of rags to be folded
please enter the number of rags to be ironed
ple
--------------------------------------------------------------------------------

🧠 Original: difference between ai and ml
✨ Refined:  the difference between artificial intelligence (ai) and machine learning (ml)

### Explanation:
the difference between ai and ml is that ai is a branch of computer science that focuses on the development of intelligent machines, while ml is a branch of computer science that focuses on the development of machines that can learn and make decisions based on data.
--------------------------------------------------------------------------------

🧠 Original: importance of tokenization
✨ Refined:  Input: Importance of tokenization
----------------------

In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model="tinyllama-improm2prompt-v1",
    tokenizer="tinyllama-improm2prompt-v1",
    torch_dtype=torch.float16,
    device_map="auto",
)
def refine_prompt(raw_prompt):
    template = (
        "### Instruction:\n"
        "You are a prompt-engineering assistant. "
        "Rewrite the given user prompt into a clear, structured, and detailed instruction "
        "related to AI, data structures, algorithms, or programming. "
        "Do NOT include questions, interactive steps, or literal actions.\n\n"
        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )
    output = pipe(
        template,
        max_new_tokens=100,
        do_sample=False,
        num_beams=8,
        early_stopping=True,
        eos_token_id=pipe.tokenizer.eos_token_id,
        return_full_text=False,
    )[0]["generated_text"]

    refined = output.split("### Output:")[-1].strip().split("### Instruction:")[0].strip()
    return refined

prompts = [
    "rag system",
    "difference between ai and ml",
    "importance of tokenization",
    "generate creative writing ideas",
    "how to fine-tune a model",
]

for p in prompts:
    print(f"\n🧠 Original: {p}\n✨ Refined:  {refine_prompt(p)}\n{'-'*80}")



🧠 Original: rag system
✨ Refined:  Write a clear, structured, and detailed instruction related to AI, data structures, algorithms, or programming that takes the user through a step-by-step process of setting up a Rag system. Ensure that the instruction is concise, easy to follow, and includes any necessary prerequisites or dependencies.
--------------------------------------------------------------------------------

🧠 Original: difference between ai and ml
✨ Refined:  Write a clear, structured, and detailed instruction related to AI, data structures, algorithms, or programming that explains the difference between artificial intelligence (ai) and machine learning (ml). Do NOT include questions, interactive steps, or literal actions.
--------------------------------------------------------------------------------

🧠 Original: importance of tokenization
✨ Refined:  Write a clear, structured, and detailed instruction related to AI, data structures, algorithms, or programming that explain

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving refinement_dataset_v2_200.json to refinement_dataset_v2_200.json


In [ ]:
import json

with open("formatted_tinyllama_dataset.json") as f:
    base = json.load(f)
with open("refinement_dataset_v2_200.json") as f:
    add = json.load(f)

merged = base + add
with open("merged_refiner_dataset.json", "w") as f:
    json.dump(merged, f, indent=2)


In [ ]:
# === Imports ===
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

# === Load merged dataset ===
dataset = load_dataset("json", data_files="merged_refiner_dataset.json")
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

# === Model setup ===
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
output_model = "tinyllama-improm2prompt-v3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# === LoRA configuration ===
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

# === Training arguments ===
training_arguments = TrainingArguments(
    output_dir=output_model,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,   # lower LR for stable continued training
    num_train_epochs=2,   # just 2 epochs to "lock in" improvements
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",     # disable wandb
)

# === Trainer ===
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    args=training_arguments,
    packing=False,
    max_seq_length=1024,
)

# === Train ===
trainer.train()

# === Save fine-tuned model ===
trainer.save_model(output_model)
print(f"✅ Retraining complete! Model saved to {output_model}")


Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/503 [00:00<?, ? examples/s]

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

Step,Training Loss
10,3.006900
20,2.674600
30,2.429000
40,2.113900
50,1.838700
60,1.581200
70,1.430100
80,1.268100
90,1.188800
100,1.140100


✅ Retraining complete! Model saved to tinyllama-improm2prompt-v3


In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="tinyllama-improm2prompt-v3",
    tokenizer="tinyllama-improm2prompt-v3",
    torch_dtype=torch.float16,
    device_map="auto",
)

def refine_prompt(raw_prompt):
    template = (
        "### Instruction:\n"
        "Refine the following user prompt into a precise, professional, and contextually rich instruction "
        "that could be used to guide an AI system. "
        "Focus on clarity and completeness — do NOT describe how to perform the task, "
        "just rewrite the prompt to make it clearer.\n\n"
        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )
    output = pipe(
        template,
        max_new_tokens=100,
        do_sample=False,
        num_beams=6,
        early_stopping=True,
        eos_token_id=pipe.tokenizer.eos_token_id,
        return_full_text=False,
    )[0]["generated_text"]

    refined = output.split("### Output:")[-1].strip().split("### Instruction:")[0].strip()
    return refined

# Test
prompts = [
    "rag system",
    "difference between ai and ml",
    "importance of tokenization",
    "graph traversal example",
    "dynamic programming explanation"
]

for p in prompts:
    print(f"\n🧠 Original: {p}\n✨ Refined: {refine_prompt(p)}\n{'-'*80}")



🧠 Original: rag system
✨ Refined: Please enter the name of the file you want to rename:
Enter the name of the file you want to rename:
Enter the name of the file you want to rename:
Enter the name of the file you want to rename:
Enter the name of the file you want to rename:
--------------------------------------------------------------------------------

🧠 Original: difference between ai and ml
✨ Refined: 1. Define the difference between artificial intelligence (ai) and machine learning (ml).
2. Explain how ai and ml work together to create intelligent systems.
3. Discuss the advantages and disadvantages of using ai and ml in various industries, such as healthcare, finance, and marketing.
4. Provide examples of successful ai and ml applications in these industries.
5. Explain how ai and ml can be integrated
--------------------------------------------------------------------------------

🧠 Original: importance of tokenization
✨ Refined: Provide a step-by-step guide on how to perform 

In [ ]:
refine_prompt("how to implement bfs")


NameError: name 'refine_prompt' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!ls "/content/drive/MyDrive/"


Mounted at /content/drive
'Carnival Logos.pptx'
'chemical effects of electric current NCERT Question answer.pdf'
'CLASS 8- SOIL CONSERVARTION.gdoc'
'Class8 worksheet sound 2019-20.pdf'
 Classroom
'Class VIII- Minerals and Power Resources.pdf'
'Colab Notebooks'
'English Report.pptx'
 IMG_20230309_135531.jpg
'INFORMAL LETTER ENG.docx'
 Maths
 OOPS_DA1_24BDE0027.pdf
'Operations Subdomain.gdoc'
 Riviera
 Screenshot_20230715-150746.jpg
'shiv nadar application form.pdf'
'Sponsorship Domain.gdoc'
 srujan.mp3
 Tech_eng
'Triangle .pdf'
'Untitled Jam (1).pdf'
'Untitled Jam (2).pdf'
'Untitled Jam.pdf'
'Untitled presentation.gslides'
'WhatsApp Chat with Apj homies'
'WhatsApp Chat with Family'
'WhatsApp Chat with Hi'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

# === Load merged dataset ===
dataset = load_dataset("json", data_files="merged_refiner_dataset.json")
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

# === Model setup ===
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
output_model = "tinyllama-improm2prompt-v3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

# === LoRA configuration ===
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

# === Training arguments ===
training_arguments = TrainingArguments(
    output_dir=output_model,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=2,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)

# === Trainer ===
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    args=training_arguments,
    packing=False,
    max_seq_length=1024,
)

# === Train ===
trainer.train()

# === Save model in Colab ===
trainer.save_model(output_model)
print("✅ Model saved in Colab")


ModuleNotFoundError: No module named 'trl'

In [ ]:
!pip install -U transformers==4.44.2 trl==0.9.6 peft==0.10.0 accelerate==0.33.0 bitsandbytes==0.43.3


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: tokenizers
    Found existing i

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer


ModuleNotFoundError: No module named 'triton.ops'

In [ ]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [ ]:
!pip uninstall -y bitsandbytes triton
!pip install bitsandbytes==0.43.2
!pip install triton==2.1.0


Found existing installation: bitsandbytes 0.43.3
Uninstalling bitsandbytes-0.43.3:
  Successfully uninstalled bitsandbytes-0.43.3
Found existing installation: triton 3.4.0
Uninstalling triton-3.4.0:
  Successfully uninstalled triton-3.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 7.2 MB/s eta 0:00:00


ERROR: Could not find a version that satisfies the requirement triton==2.1.0 (from versions: 2.2.0, 2.3.0, 2.3.1, 3.0.0, 3.1.0, 3.2.0, 3.3.0, 3.3.1, 3.4.0, 3.5.0)
ERROR: No matching distribution found for triton==2.1.0


In [ ]:
import bitsandbytes as bnb
print("GPU detected:", bnb.cuda_setup.is_available())


ModuleNotFoundError: No module named 'triton.ops'

In [ ]:
!pip uninstall -y bitsandbytes triton
!pip install bitsandbytes-cuda120==0.43.1
!pip install triton==2.1.0


Found existing installation: bitsandbytes 0.43.2
Uninstalling bitsandbytes-0.43.2:
  Successfully uninstalled bitsandbytes-0.43.2
Found existing installation: triton 3.4.0
Uninstalling triton-3.4.0:
  Successfully uninstalled triton-3.4.0
ERROR: Could not find a version that satisfies the requirement bitsandbytes-cuda120==0.43.1 (from versions: none)
ERROR: No matching distribution found for bitsandbytes-cuda120==0.43.1
ERROR: Could not find a version that satisfies the requirement triton==2.1.0 (from versions: 2.2.0, 2.3.0, 2.3.1, 3.0.0, 3.1.0, 3.2.0, 3.3.0, 3.3.1, 3.4.0, 3.5.0)
ERROR: No matching distribution found for triton==2.1.0


In [ ]:
!pip uninstall -y bitsandbytes triton
!pip install -U bitsandbytes==0.43.3 triton==2.3.0


  Using cached bitsandbytes-0.43.3-py3-none-manylinux_2_24_x86_64.whl.metadata (3.5 kB)
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
Using cached bitsandbytes-0.43.3-py3-none-manylinux_2_24_x86_64.whl (137.5 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 852.6 

In [ ]:
import torch, bitsandbytes as bnb, sys, site, os, subprocess, importlib

print("Torch CUDA available:", torch.cuda.is_available())
print("Torch CUDA version (reported by nvcc):")
print(subprocess.getoutput("nvcc --version"))

# Quick attempt to import (may warn about missing libbitsandbytes_cuda126.so)
try:
    _ = bnb.__version__
    print("bitsandbytes version:", bnb.__version__)
except Exception as e:
    print("bitsandbytes import error:", e)


Torch CUDA available: True
Torch CUDA version (reported by nvcc):
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
bitsandbytes version: 0.43.3


In [ ]:
import bitsandbytes as bnb
print("GPU support:", bnb.cuda_setup.is_available())


AttributeError: module 'bitsandbytes' has no attribute 'cuda_setup'

In [ ]:
import bitsandbytes as bnb
from bitsandbytes.cuda_setup import cuda_setup
print("GPU support:", cuda_setup.is_available())


ModuleNotFoundError: No module named 'bitsandbytes.cuda_setup'

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

try:
    test_bnb = AutoModelForCausalLM.from_pretrained(
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True),
        device_map="auto",
    )
    print("✅ 4-bit quantization works — GPU support confirmed.")
except Exception as e:
    print("❌ 4-bit load failed. Error:")
    print(e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ 4-bit quantization works — GPU support confirmed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from datasets import load_dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

# Load dataset
dataset = load_dataset("json", data_files="merged_refiner_dataset.json")
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

# Model ID + output folder
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
output_model = "tinyllama-improm2prompt-v3"

# 4-bit quantization config (correct version)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,   # ✅ Restored
)

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False
model.config.pretraining_tp = 1       # ✅ Restored

# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

# Training settings (restored LR)
training_arguments = TrainingArguments(
    output_dir=output_model,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,        # ✅ Restored learning rate
    num_train_epochs=2,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    args=training_arguments,
    packing=False,
    max_seq_length=1024,
)

trainer.train()


FileNotFoundError: Unable to find '/content/merged_refiner_dataset.json'

In [ ]:
from google.colab import files
uploaded = files.upload()


Saving formatted_tinyllama_dataset.json to formatted_tinyllama_dataset.json
Saving refinement_dataset_v2_200.json to refinement_dataset_v2_200.json


In [ ]:
import json

with open("formatted_tinyllama_dataset.json") as f:
    base = json.load(f)
with open("refinement_dataset_v2_200.json") as f:
    add = json.load(f)

merged = base + add

with open("merged_refiner_dataset.json", "w") as f:
    json.dump(merged, f, indent=2)

print("✅ merged_refiner_dataset.json created successfully!")


✅ merged_refiner_dataset.json created successfully!


In [ ]:
!ls -lh merged_refiner_dataset.json


-rw-r--r-- 1 root root 191K Nov  5 07:38 merged_refiner_dataset.json


In [ ]:
from datasets import load_dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

# Load dataset
dataset = load_dataset("json", data_files="merged_refiner_dataset.json")
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

# Model ID + output folder
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
output_model = "tinyllama-improm2prompt-v3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False
model.config.pretraining_tp = 1

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

training_arguments = TrainingArguments(
    output_dir=output_model,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    num_train_epochs=2,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    args=training_arguments,
    packing=False,
    max_seq_length=1024,
)

trainer.train()


Generating train split: 0 examples [00:00, ? examples/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/503 [00:00<?, ? examples/s]

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

Step,Training Loss
10,3.009900
20,2.682400
30,2.441600
40,2.128500
50,1.860400
60,1.603200
70,1.450600
80,1.285600
90,1.207700
100,1.159200


TrainOutput(global_step=126, training_loss=1.7121105988820393, metrics={'train_runtime': 64.3121, 'train_samples_per_second': 15.642, 'train_steps_per_second': 1.959, 'total_flos': 569047233011712.0, 'train_loss': 1.7121105988820393, 'epoch': 2.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/tinyllama-improm2prompt-v3 /content/drive/MyDrive/
print("✅ Model saved permanently to Google Drive!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Model saved permanently to Google Drive!


In [ ]:
from transformers import pipeline

model_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3"

pipe = pipeline(
    "text-generation",
    model=model_path,
    tokenizer=model_path,
    torch_dtype="auto",
    device_map="auto",
)


RuntimeError: Failed to import transformers.pipelines because of the following error (look up to see its traceback):
module 'torch.library' has no attribute 'register_fake'

In [ ]:
!pip uninstall -y transformers trl peft accelerate


Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
Found existing installation: trl 0.9.6
Uninstalling trl-0.9.6:
  Successfully uninstalled trl-0.9.6
Found existing installation: peft 0.10.0
Uninstalling peft-0.10.0:
  Successfully uninstalled peft-0.10.0
Found existing installation: accelerate 0.33.0
Uninstalling accelerate-0.33.0:
  Successfully uninstalled accelerate-0.33.0


In [ ]:
!pip install -U transformers==4.36.2
!pip install -U trl==0.7.10
!pip install -U peft==0.8.2
!pip install -U accelerate==0.25.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 106.5 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.2 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.9/150.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 35.8 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 15.7 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0


In [ ]:
from transformers import pipeline

model_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3"

pipe = pipeline(
    "text-generation",
    model=model_path,
    tokenizer=model_path,
    device_map="auto",
    torch_dtype="auto",
)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


OSError: /content/drive/MyDrive/tinyllama-improm2prompt-v3 does not appear to have a file named config.json. Checkout 'https://huggingface.co//content/drive/MyDrive/tinyllama-improm2prompt-v3/None' for available files.

In [ ]:
!ls -R "/content/drive/MyDrive/tinyllama-improm2prompt-v3"


/content/drive/MyDrive/tinyllama-improm2prompt-v3:
checkpoint-126

/content/drive/MyDrive/tinyllama-improm2prompt-v3/checkpoint-126:
adapter_config.json	   rng_state.pth	    tokenizer.json
adapter_model.safetensors  scheduler.pt		    tokenizer.model
optimizer.pt		   special_tokens_map.json  trainer_state.json
README.md		   tokenizer_config.json    training_args.bin


In [ ]:
!pip install -U transformers==4.36.2 peft==0.8.2 trl==0.7.10 accelerate==0.25.0


In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3/checkpoint-126"

# Load tokenizer from base model
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token

# Load base model + apply LoRA adapter
model = AutoPeftModelForCausalLM.from_pretrained(
    lora_path,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Merge LoRA weights into the base model permanently
model = model.merge_and_unload()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


TypeError: LoraConfig.__init__() got an unexpected keyword argument 'layer_replication'

In [ ]:
!pip install -U peft==0.10.0


  Using cached peft-0.10.0-py3-none-any.whl.metadata (13 kB)
Using cached peft-0.10.0-py3-none-any.whl (199 kB)
  Attempting uninstall: peft
    Found existing installation: peft 0.8.2
    Uninstalling peft-0.8.2:
      Successfully uninstalled peft-0.8.2


In [ ]:
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [ ]:
base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3/checkpoint-126"

tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token

model = AutoPeftModelForCausalLM.from_pretrained(
    lora_path,
    device_map="auto",
    torch_dtype="auto",
)

model = model.merge_and_unload()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
save_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Fully merged model saved at:", save_path)


✅ Fully merged model saved at: /content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED


In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=save_path,
    tokenizer=save_path,
    device_map="auto",
    torch_dtype="auto",
)


In [ ]:
import torch
from transformers import pipeline

save_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED"   # ✅ full merged model folder

pipe = pipeline(
    "text-generation",
    model=save_path,
    tokenizer=save_path,
    torch_dtype=torch.float16,
    device_map="auto",
)

def refine_prompt(raw_prompt):
    template = (
        "### Instruction:\n"
        "Refine the following user prompt into a precise, professional, and contextually rich instruction "
        "that could be used to guide an AI system. "
        "Focus on clarity and completeness — do NOT describe how to perform the task, "
        "just rewrite the prompt to make it clearer.\n\n"
        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )

    output = pipe(
        template,
        max_new_tokens=100,
        do_sample=False,
        num_beams=6,
        early_stopping=True,
        eos_token_id=pipe.tokenizer.eos_token_id,
        return_full_text=False,
    )[0]["generated_text"]

    refined = output.strip()
    return refined


# ✅ Test
prompts = [
    "rag system",
    "difference between ai and ml",
    "importance of tokenization",
    "graph traversal example",
    "dynamic programming explanation"
]

for p in prompts:
    print(f"\n🧠 Original: {p}\n✨ Refined: {refine_prompt(p)}\n{'-'*80}")



🧠 Original: rag system
✨ Refined: Refine the user prompt into a precise, professional, and contextually rich instruction that could be used to guide an AI system.
--------------------------------------------------------------------------------

🧠 Original: difference between ai and ml
✨ Refined: Refine the user prompt into a precise, professional, and contextually rich instruction that could be used to guide an AI system. Focus on clarity and completeness.
--------------------------------------------------------------------------------

🧠 Original: importance of tokenization
✨ Refined: Refine the user prompt into a precise, professional, and contextually rich instruction that could be used to guide an AI system. Focus on clarity and completeness.
--------------------------------------------------------------------------------

🧠 Original: graph traversal example
✨ Refined: Refine the user prompt into a precise, professional, and contextually rich instruction that could be used to guid

In [ ]:
from transformers import pipeline
import torch

save_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED"  # merged model folder

pipe = pipeline(
    "text-generation",
    model=save_path,
    tokenizer=save_path,
    torch_dtype=torch.float16,
    device_map="auto",
)

FEWSHOT = (
    "### Instruction:\n"
    "Refine the following user prompt into a precise, professional, and contextually rich instruction "
    "for an AI system. Do NOT give steps or answers—only rewrite the prompt.\n\n"

    # 1) AI example
    "### Input:\nrag system\n\n"
    "### Output:\nExplain the architecture and workflow of a Retrieval-Augmented Generation (RAG) system, "
    "including how the retriever selects relevant documents and how the generator integrates retrieved context into its response.\n\n"

    # 2) DSA example
    "### Input:\nbinary search tree\n\n"
    "### Output:\nDescribe the properties and core operations of a Binary Search Tree (BST)—insertion, search, and in-order traversal—with a concise illustrative example.\n\n"

    # 3) DSA example
    "### Input:\ndynamic programming example\n\n"
    "### Output:\nFormulate a clear instruction that requests a dynamic-programming solution outline, including subproblem definition, recurrence relation, and base cases, using a representative problem (e.g., coin change or LIS).\n\n"
)

def refine_prompt(raw_prompt: str) -> str:
    prompt = (
        FEWSHOT +
        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )
    out = pipe(
        prompt,
        max_new_tokens=96,
        do_sample=False,          # deterministic
        num_beams=6,              # search for best non-generic continuation
        length_penalty=0.9,
        early_stopping=True,
        no_repeat_ngram_size=3,   # reduce boilerplate loops
        return_full_text=False,
        eos_token_id=pipe.tokenizer.eos_token_id,
    )[0]["generated_text"].strip()

    # Post-clean: stop if model drifts back into a new "###" section
    out = out.split("### Instruction:")[0].strip()

    # If it still returns a meta sentence, try a second pass with a tighter one-shot
    if out.lower().startswith("refine the user prompt") or out.lower().startswith("write a clear, structured"):
        one_shot = (
            "### Instruction:\nRewrite the prompt so it becomes a concrete, domain-specific instruction. "
            "No steps, no answers—just the improved instruction.\n\n"
            f"### Input:\n{raw_prompt}\n\n### Output:\n"
        )
        out2 = pipe(
            one_shot,
            max_new_tokens=80,
            do_sample=False,
            num_beams=8,
            no_repeat_ngram_size=3,
            return_full_text=False,
            eos_token_id=pipe.tokenizer.eos_token_id,
        )[0]["generated_text"].strip()
        out2 = out2.split("### Instruction:")[0].strip()
        if len(out2) > 0:
            out = out2
    return out


HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED'. Use `repo_type` argument if needed.

In [ ]:
!ls "/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED"


ls: cannot access '/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED': No such file or directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED"


config.json		special_tokens_map.json  tokenizer.model
generation_config.json	tokenizer_config.json
model.safetensors	tokenizer.json


In [ ]:
save_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED"


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained(save_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    save_path,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True,
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
)


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


In [ ]:
def refine_prompt(raw_prompt):
    prompt = (
        "Rewrite the following prompt so that it becomes a clearer, more specific instruction. "
        "Do NOT answer it, only rewrite it.\n\n"
        f"Original Prompt: {raw_prompt}\n"
        "Refined Prompt:"
    )

    result = pipe(prompt, max_new_tokens=80, do_sample=False)[0]["generated_text"]
    return result.replace(prompt, "").strip()


In [ ]:
refine_prompt("bfs")


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'Implement a breadth-first search algorithm in C++.\n\nOriginal Explanation: breadth-first search (BFS) is a search algorithm that traverses the graph in a breadth-first manner. It is a popular algorithm for finding paths in a graph. In this exercise, you will implement BFS in C++.\n\nRefined Explanation: B'

In [ ]:
def refine_prompt(raw_prompt):
    template = (
        # === Few-shot examples to guide refinement ===
        "### Instruction:\n"
        "Refine the following user prompt into a precise, professional, and contextually rich instruction "
        "that could be used to guide an AI system. "
        "Focus on clarity and completeness — do NOT describe how to perform the task, "
        "just rewrite the prompt to make it clearer.\n\n"

        "### Input:\nrag system\n\n"
        "### Output:\nExplain the structure and workflow of a Retrieval-Augmented Generation (RAG) system, "
        "including how external knowledge retrieval is integrated into the generation process.\n\n"

        "### Input:\ndifference between ai and ml\n\n"
        "### Output:\nExplain the conceptual and functional differences between Artificial Intelligence (AI) "
        "and Machine Learning (ML), highlighting how ML fits as a subset within AI.\n\n"

        "### Input:\nimportance of tokenization\n\n"
        "### Output:\nDescribe the role of tokenization in natural language processing, "
        "including how text is segmented into meaningful units and why this step impacts model accuracy.\n\n"

        # === Your actual input goes here ===
        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )

    output = pipe(
        template,
        max_new_tokens=120,
        do_sample=False,
        num_beams=6,
        early_stopping=True,
        eos_token_id=pipe.tokenizer.eos_token_id,
        return_full_text=False,
    )[0]["generated_text"]

    refined = (
        output.split("### Output:")[-1]
        .split("### Instruction:")[0]
        .strip()
    )
    return refined


In [ ]:
refine_prompt("bfs")

NameError: name 'pipe' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
save_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED"

In [ ]:
model_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED"


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True,
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
)


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


In [ ]:
def refine_prompt(raw_prompt):
    template = (
        "### Instruction:\n"
        "Refine the following user prompt into a precise, professional, and contextually rich instruction "
        "that could be used to guide an AI system. "
        "Focus on clarity and completeness — do NOT describe how to perform the task, "
        "just rewrite the prompt to make it clearer.\n\n"

        "### Input:\nrag system\n\n"
        "### Output:\nExplain the architecture and workflow of a Retrieval-Augmented Generation (RAG) system, "
        "detailing how external knowledge retrieval is integrated into the response generation pipeline.\n\n"

        "### Input:\ndifference between ai and ml\n\n"
        "### Output:\nDescribe the conceptual and practical distinctions between Artificial Intelligence (AI) "
        "and Machine Learning (ML), noting the hierarchical relationship between the two.\n\n"

        "### Input:\nimportance of tokenization\n\n"
        "### Output:\nExplain the significance of tokenization in natural language processing, including how "
        "text is segmented into interpretable units and why this affects model understanding and accuracy.\n\n"

        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )

    output = pipe(
        template,
        max_new_tokens=120,
        do_sample=False,
        num_beams=6,
        early_stopping=True,
        return_full_text=False,
        eos_token_id=pipe.tokenizer.eos_token_id,
    )[0]["generated_text"]

    refined = output.split("### Output:")[-1].split("### Instruction:")[0].strip()
    return refined


In [ ]:
print(refine_prompt("bfs"))


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Define Breadth-First Search (BFS) and explain how it differs from Depth-First Search (DFS) in the context of graph traversal algorithms.


In [ ]:
print(refine_prompt("greedy algorithm"))


Describe the Greedy Algorithm used in Natural Language Processing (NLP), including how it works and how it differs from other algorithms in the field.


In [ ]:
print(refine_prompt("rag system"))

Explain the architecture and workflow of a Retrieval-Augmented Generation (RAG) system, detailing how external knowledge retrieval is integrated into the response generation pipeline.


In [ ]:
print(refine_prompt("array"))

Provide an in-depth explanation of an array, including its elements, properties, and how it is used in programming and data analysis.


In [ ]:
print(refine_prompt("infix expression"))

Describe the syntax and semantics of infix expressions, including how they differ from prefix and postfix expressions, and how they are used in programming languages.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
model_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True,
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
)


`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


In [ ]:
def refine_prompt(raw_prompt):
    template = (
        "### Instruction:\n"
        "Refine the following user prompt into a precise, professional, and contextually rich instruction "
        "that could be used to guide an AI system. "
        "Focus on clarity and completeness — do NOT describe how to perform the task, "
        "just rewrite the prompt to make it clearer.\n\n"

        "### Input:\nrag system\n\n"
        "### Output:\nExplain the architecture and workflow of a Retrieval-Augmented Generation (RAG) system, "
        "detailing how external knowledge retrieval is integrated into the response generation pipeline.\n\n"

        "### Input:\ndifference between ai and ml\n\n"
        "### Output:\nDescribe the conceptual and practical distinctions between Artificial Intelligence (AI) "
        "and Machine Learning (ML), noting the hierarchical relationship between the two.\n\n"

        "### Input:\nimportance of tokenization\n\n"
        "### Output:\nExplain the significance of tokenization in natural language processing, including how "
        "text is segmented into interpretable units and why this affects model understanding and accuracy.\n\n"

        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )

    output = pipe(
        template,
        max_new_tokens=120,
        do_sample=False,
        num_beams=6,
        early_stopping=True,
        return_full_text=False,
        eos_token_id=pipe.tokenizer.eos_token_id,
    )[0]["generated_text"]

    refined = output.split("### Output:")[-1].split("### Instruction:")[0].strip()
    return refined


In [ ]:
print(refine_prompt("bfs"))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Define Breadth-First Search (BFS) and explain how it differs from Depth-First Search (DFS) in the context of graph traversal algorithms.


In [ ]:
def show_refinement(prompt):
    refined = refine_prompt(prompt)
    print(f"🧠 Original: {prompt}\n✨ Refined: {refined}\n" + "-"*80)


In [ ]:
def run_refiner():
    while True:
        user_input = input("Enter a prompt to refine (or type 'exit' to stop): ")

        if user_input.lower().strip() in ["exit", "quit", "q"]:
            print("👋 Exiting Prompt Refiner. Have a great day!")
            break

        show_refinement(user_input)


In [ ]:
run_refiner()


Enter a prompt to refine (or type 'exit' to stop): bfs
🧠 Original: bfs
✨ Refined: Define Breadth-First Search (BFS) and explain how it differs from Depth-First Search (DFS) in the context of graph traversal algorithms.
--------------------------------------------------------------------------------
Enter a prompt to refine (or type 'exit' to stop): greedy algorithm
🧠 Original: greedy algorithm
✨ Refined: Describe the Greedy Algorithm used in Natural Language Processing (NLP), including how it works and how it differs from other algorithms in the field.
--------------------------------------------------------------------------------
Enter a prompt to refine (or type 'exit' to stop): rag system
🧠 Original: rag system
✨ Refined: Explain the architecture and workflow of a Retrieval-Augmented Generation (RAG) system, detailing how external knowledge retrieval is integrated into the response generation pipeline.
--------------------------------------------------------------------------------

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
model_path = "/content/drive/MyDrive/tinyllama-improm2prompt-v3-MERGED"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    local_files_only=True,
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto",
)

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


In [ ]:
def refine_prompt(raw_prompt):
    template = (
        "### Instruction:\n"
        "Refine the following user prompt into a precise, professional, and contextually rich instruction "
        "that could be used to guide an AI system. "
        "Focus on clarity and completeness — do NOT describe how to perform the task, "
        "just rewrite the prompt to make it clearer.\n\n"

        "### Input:\nrag system\n\n"
        "### Output:\nExplain the architecture and workflow of a Retrieval-Augmented Generation (RAG) system, "
        "detailing how external knowledge retrieval is integrated into the response generation pipeline.\n\n"

        "### Input:\ndifference between ai and ml\n\n"
        "### Output:\nDescribe the conceptual and practical distinctions between Artificial Intelligence (AI) "
        "and Machine Learning (ML), noting the hierarchical relationship between the two.\n\n"

        "### Input:\nimportance of tokenization\n\n"
        "### Output:\nExplain the significance of tokenization in natural language processing, including how "
        "text is segmented into interpretable units and why this affects model understanding and accuracy.\n\n"

        f"### Input:\n{raw_prompt}\n\n### Output:\n"
    )

    output = pipe(
        template,
        max_new_tokens=120,
        do_sample=False,
        num_beams=6,
        early_stopping=True,
        return_full_text=False,
        eos_token_id=pipe.tokenizer.eos_token_id,
    )[0]["generated_text"]

    refined = output.split("### Output:")[-1].split("### Instruction:")[0].strip()
    return refined


In [ ]:
def show_refinement(prompt):
    refined = refine_prompt(prompt)
    print(f"🧠 Original: {prompt}\n✨ Refined: {refined}\n" + "-"*80)

In [ ]:
def run_refiner():
    while True:
        user_input = input("Enter a prompt to refine (or type 'exit' to stop): ")

        if user_input.lower().strip() in ["exit", "quit", "q"]:
            print("👋 Exiting Prompt Refiner. Have a great day!")
            break

        show_refinement(user_input)

In [ ]:
run_refiner()


Enter a prompt to refine (or type 'exit' to stop): bfs


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🧠 Original: bfs
✨ Refined: Define Breadth-First Search (BFS) and explain how it differs from Depth-First Search (DFS) in the context of graph traversal algorithms.
--------------------------------------------------------------------------------
Enter a prompt to refine (or type 'exit' to stop): exit
👋 Exiting Prompt Refiner. Have a great day!
